# Research Agent API - Synchronous Client

A robust Python client for the [Bigdata.com Research Agent API](https://docs.bigdata.com/how-to-guides/agents) that consumes the Server-Sent Events stream and hands back one finished result: the answer, correctly attributed citations, and any charts the agent produced.

## Features

| Feature | Description |
|---------|-------------|
| **Synchronous interface** | One blocking call - no async/await |
| **Complete message coverage** | Every public SSE message type is handled; unknown types are ignored so new API versions do not break the client |
| **Accurate citations** | Document citations and whole-tool attributions are handled separately, so a reference never renders as a blank `N/A` entry |
| **Typed errors** | HTTP status codes and in-stream `ERROR` events map to specific exception classes |
| **Automatic retries** | Exponential backoff with jitter for `429`, `5xx`, and transient network failures |
| **Charts** | `CHART` events are collected as Vega-Lite specs anchored to answer offsets |
| **Conversation continuity** | `chat_id` follow-ups and `checkpoint_id` branching |

## Reference documentation

- [Concepts overview](https://docs.bigdata.com/how-to-guides/agents/concepts/overview)
- [Streaming responses](https://docs.bigdata.com/how-to-guides/agents/concepts/streaming-responses)
- [Grounding and citations](https://docs.bigdata.com/how-to-guides/agents/concepts/grounding-and-citations)
- [Code execution and charts](https://docs.bigdata.com/how-to-guides/agents/concepts/code-execution-and-charts)
- [Error handling](https://docs.bigdata.com/how-to-guides/agents/concepts/error-handling)

---
## 1. Setup

The client needs `requests` and a `BIGDATA_API_KEY`. Everything else is standard library.

In [37]:
import json
import logging
import os

from IPython.display import Markdown, display

from research_client import ResearchClient, format_source_date, setup_logging

os.makedirs("output", exist_ok=True)

# Logging is flushed on every record, so the log file survives a hard failure
# mid-stream. Set console=False to keep the notebook output quiet.
setup_logging(
    log_file="output/research_client.log",
    level=logging.INFO,
    console=False,
    file_mode="w",
)

print("Ready.")

Ready.


---
## 2. Configure the client

Retry, timeout, and tool settings are all set once on the client. Per-request overrides are available on `research()`.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `timeout` | `300` | Connection timeout in seconds |
| `stream_timeout` | `60.0` | Max seconds to wait between SSE chunks before treating the connection as stalled |
| `max_retries` | `3` | Retry attempts for transient failures |
| `retry_delay` | `1.0` | Initial backoff delay |
| `retry_backoff` | `2.0` | Exponential backoff multiplier |
| `retry_max_delay` | `60.0` | Upper bound on the backoff delay |
| `persistence_mode` | `"enabled"` | Saves conversation history so `chat_id` follow-ups work. The API itself defaults to `"disabled"` |
| `code_execution` | `None` | Whether the agent may run sandboxed Python. `None` keeps the server default (on) |
| `chart_generation` | `None` | Whether the agent may emit `CHART` events. `None` keeps the server default (off) |

Backoff uses **full jitter** (`delay + random(0, delay)`), which is the pattern the
[error handling guide](https://docs.bigdata.com/how-to-guides/agents/concepts/error-handling#retry-and-backoff) recommends for `429` and `5xx`.

In [38]:
client = ResearchClient(
    # Timeouts
    timeout=300,
    stream_timeout=90.0,
    # Retries
    max_retries=3,
    retry_delay=2.0,
    retry_backoff=2.0,
    retry_max_delay=60.0,
    # Conversation history, required for follow_up()
    persistence_mode="enabled",
    # Let the agent run Python and draw charts
    code_execution=True,
    chart_generation=True,
)

print("Client configured.")

Client configured.


### Demonstrate retries

Transient failures such as a dropped connection, a `429`, or a `5xx` are retried automatically with exponential backoff and full jitter. Permanent client errors (`400`, `401`, `403`, `404`) are raised immediately.

A live outage is rare while you are reading this notebook, so the cell below **injects one retryable failure** on the first `requests.post`, then lets the real call through. Watch for the `Retryable error` / `Retry 1/3` log lines.

In [39]:
import logging
from unittest.mock import patch

from requests.exceptions import ConnectionError as RequestsConnectionError

from research_client import logger as research_logger

# Short delays so the demo finishes quickly; the main `client` above keeps the
# production-style retry profile for the rest of the notebook.
retry_demo_client = ResearchClient(
    timeout=300,
    stream_timeout=90.0,
    max_retries=3,
    retry_delay=0.5,
    retry_backoff=2.0,
    retry_max_delay=5.0,
    persistence_mode="enabled",
)

# Surface retry warnings in the notebook output for this cell only.
console = logging.StreamHandler()
console.setLevel(logging.WARNING)
console.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
research_logger.addHandler(console)

call_count = {"n": 0}
real_post = __import__("requests").post


def flaky_post(*args, **kwargs):
    """Fail once with a retryable network error, then behave normally."""
    call_count["n"] += 1
    if call_count["n"] == 1:
        raise RequestsConnectionError("simulated dropped connection (demo)")
    return real_post(*args, **kwargs)


print("Forcing a ConnectionError on attempt 1, then succeeding...\n")

with patch("requests.post", side_effect=flaky_post):
    retry_demo = retry_demo_client.research(
        message="What is the current level of the Nasdaq 100?",
        research_effort="lite",
    )

research_logger.removeHandler(console)

print(f"\nHTTP attempts: {call_count['n']}  (1 failure + 1 success)")
print(f"Answer length: {len(retry_demo.answer):,} characters")
print(f"Processing time: {retry_demo.processing_time_ms}ms")
print("Retry path exercised successfully.")

WARNING - Retryable error on attempt 1/4: ConnectionError: simulated dropped connection (demo)
WARNING - Retry 1/3 in 1.0s


Forcing a ConnectionError on attempt 1, then succeeding...


HTTP attempts: 2  (1 failure + 1 success)
Answer length: 42 characters
Processing time: 15226ms
Retry path exercised successfully.


---
## 3. Run a research query

The agent streams typed events while it works. Passing an `on_event` callback lets you show progress without dealing with the stream yourself.

A `research_effort` of `"lite"` returns in roughly 10-20 seconds; `"standard"` runs multi-step research and takes 20-60 seconds.

In [52]:
QUERY = "How is Nasdaq 100 Performing?"


def make_progress_printer():
    """
    Build an `on_event` callback that prints a compact progress line per event.

    PLANNING resends the whole plan every time it changes, so already-reported
    steps are suppressed rather than reprinted.
    """
    reported: set[str] = set()

    def show_progress(msg_type: str, msg: dict) -> None:
        if msg_type == "PLANNING":
            for step in msg.get("plan", {}).get("steps", []):
                line = f"  [{step.get('status', '').lower()}] {step.get('description')}"
                if step.get("status") in ("IN_PROGRESS", "COMPLETED") and line not in reported:
                    reported.add(line)
                    print(line)
        elif msg_type == "ACTION":
            print(f"  action: {msg.get('tool_name')}")
        elif msg_type == "CHART":
            print(f"  chart: {msg.get('title')} ({msg.get('chart_type')})")
        elif msg_type == "TOOL_ERROR":
            print(f"  tool error: {msg.get('tool_name')} - {msg.get('error')}")
        elif msg_type == "LLM_RETRY":
            print(f"  agent retrying: {msg.get('message')}")

    return show_progress


print(f"Researching: {QUERY}\n")

result = client.research(
    message=QUERY,
    research_effort="standard",
    on_event=make_progress_printer(),
)

print(f"\nDone in {result.processing_time_ms / 1000:.1f}s")
print(f"  answer:         {len(result.answer):,} characters")
print(f"  documents:      {len(result.citations)}")
print(f"  grounded spans: {len(result.grounding_refs)}")
print(f"  charts:         {len(result.charts)}")
print(f"  tool errors:    {result.tool_errors or 'none'}")
print(f"  chat_id:        {result.chat_id}")
print(f"  checkpoint_id:  {result.checkpoint_id}")

Researching: How is Nasdaq 100 Performing?

  action: search
  action: get_market_tearsheet
  [completed] Gather current Nasdaq 100 market snapshot, recent trends, and drivers (last 3-6 months).
  [completed] Identify top contributors/detractors to Nasdaq 100 performance and investigate their catalysts.
  [in_progress] Synthesize findings into a final report.
  [completed] Synthesize findings into a final report.

Done in 41.4s
  answer:         3,601 characters
  documents:      19
  grounded spans: 19
  charts:         0
  tool errors:    none
  chat_id:        1785344418-7bb27b02-175a-4069-9841-6afbb15de62b
  checkpoint_id:  1f18b6f1-05e6-6856-8010-6869df13201d


---
## 4. Answer with inline citations

`GROUNDING` events attribute spans of the answer to their sources using `start`/`end` character offsets into the **cumulative** answer text. The client buffers every `ANSWER` chunk verbatim and only resolves those offsets once the stream is complete, which is the rule the [grounding guide](https://docs.bigdata.com/how-to-guides/agents/concepts/grounding-and-citations#the-buffering-rule) sets out.

Two kinds of reference come back, and they need different rendering:

| `source` | Meaning | Rendered as |
|---|---|---|
| Populated | A document returned by the **search** tool | Headline, publisher, date, and URL |
| Absent | Every **other** tool (market tearsheet, earnings calendar, code execution) grounds the span at the whole-tool level via `audit_id` | The tool's audit title, e.g. *Market Tearsheet* |

Treating the second kind as a document is what produced blank `N/A` citations. The client now resolves it against the matching `AUDIT` trace instead.

In [53]:
answer_with_citations = result.get_answer_with_citations()
numbered_citations = result.get_numbered_citations()

display(Markdown("## Answer\n\n" + answer_with_citations))

## Answer

### Executive Summary
The Nasdaq 100 is currently experiencing a period of significant volatility and has officially slipped into correction territory, having fallen more than 10% from its record high set in early June 2026 [1]. While the broader market has shown resilience, the Nasdaq 100 faces headwinds from a sharp rotation out of the semiconductor and AI-hardware sectors, which had previously been primary engines of growth. Sentiment remains cautious as investors reassess valuation premiums in light of geopolitical tensions and mixed signals regarding corporate earnings.

### Key Performance Metrics
*As of July 29, 2026*

| Metric | Value |
| :--- | :--- |
| **Current Price** | 27,460.63 |
| **Performance (YTD)** | +8.76% |
| **Performance (3-Month)** | +1.01% |
| **Performance (1-Month)** | -7.77% |
| **Performance (6-Month)** | +5.86% |
| **Performance (1-Year)** | +17.81% |

*Source: Market Data [2]*

### Drivers of Performance
The index's recent movement is characterized by a "valuation reset" within the tech sector rather than a structural bear market [3]. Several primary factors are driving this trend:

*   **Semiconductor/AI Hardware Pullback:** The index has been hit hard by a cooling in the AI-hardware complex. Concerns regarding the sustainability of heavy AI investment and rising competition from international manufacturers have led to a sharp sell-off in memory and chip stocks [1, 4, 5].
*   **Sector Concentration:** The Nasdaq 100 remains highly sensitive to its top performers. Heavy concentration in semiconductor giants that are currently undergoing a correction has exacerbated the index's downward pressure [3].
*   **Geopolitical and Macro Volatility:** Middle East tensions have intermittently driven up oil prices, reigniting fears over inflation and Fed rate expectations. While geopolitical "cooling" (e.g., resumed peace talks) has provided temporary relief, the uncertainty continues to drive volatility [6, 7, 8].
*   **Earnings Season Focus:** The market is currently laser-focused on earnings reports from "Magnificent Seven" companies. While technical indicators suggest a near-term correction, bottom-up earnings expectations remain strong, with Nasdaq 100 EPS growth projected at +43% for the year [3].

### Top Contributors/Detractors
*   **Detractors:** The memory and semiconductor sectors have been the primary drags on performance. Companies such as SanDisk Corp. (SNDK) have seen significant declines (down over 50% from its peak), and Micron Technology Inc. (MU) has lost roughly one-third of its market value [1]. Other semiconductor peers, including Intel (INTC) and various equipment manufacturers, have faced selling pressure amid broader sector weakness [3, 4].
*   **Contributors/Market Focus:** Large-cap "Big Tech" stocks (e.g., Microsoft, Meta, Alphabet) remain the fulcrum of market sentiment. These companies are currently experiencing elevated volatility as investors look for earnings catalysts to either justify current valuations or trigger further profit-taking [1, 9, 10].

### Macro Context
The macroeconomic backdrop is defined by a "wait and see" approach regarding Federal Reserve policy and inflationary pressures. Solid U.S. manufacturing and services data (Flash PMI) have provided some fundamental support to the broader economy [7]. However, the structural anxiety regarding the tech sector is evident in the divergence between tech-heavy volatility (VXN) and the broader market (VIX), which recently hit levels not seen since the dot-com era, indicating that investors are pricing in significantly more uncertainty for technology than for the rest of the market [3].

In [61]:

def print_citation(numbered_citations):
    for citation in numbered_citations:
        number = citation["number"]

        if citation.get("type") == "TOOL":
            # Whole-tool attribution: no document, so label it with the audit title.
            title = citation.get("title") or citation["tool_name"]
            parts = [
                f"**[{number}]** {title}",
                f"Bigdata.com tool &nbsp;`{citation['tool_name']}`",
            ]
            display(Markdown("<br>".join(parts) + "\n\n---"))
            continue

        source = citation.get("source") or {}
        parts = [f"**[{number}]** {citation.get('headline', '')}".rstrip()]

        meta = []
        if source.get("name"):
            meta.append(source["name"])
        if citation.get("timestamp"):
            meta.append(format_source_date(citation["timestamp"]))
        if meta:
            parts.append(" &nbsp;|&nbsp; ".join(meta))
        if citation.get("url"):
            parts.append(f"[{citation['url'][:70]}]({citation['url']})")

        excerpts = [c.get("text", "") for c in citation.get("chunks", [])[:2]]
        body = "<br>".join(parts)
        for excerpt in excerpts:
            if excerpt:
                trimmed = excerpt.replace("\n", " ")[:300]
                body += f"\n\n> *{trimmed}...*"

        display(Markdown(body + "\n\n---"))

In [62]:
display(Markdown(f"---\n## References ({len(numbered_citations)})\n"))
print_citation(numbered_citations)

---
## References (10)


**[1]** Nasdaq 100 Enters Correction, SanDisk Sinks Over 50% From Peak: Stock Market Today<br>Benzinga &nbsp;|&nbsp; Jul 28, 2026<br>[https://www.benzinga.com/node/60736942?utm_campaign=partner_feed&utm_m](https://www.benzinga.com/node/60736942?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)

> *The Nasdaq 100 officially slipped into correction territory on Tuesday, falling more than 10% from its early June record high as the AI-driven sell-off deepened on mounting concerns over rising competition from China. The memory sector has been at the heart of July's rout. SanDisk Corp. (NASDAQ:SNDK...*

---

**[2]** Market Tearsheet<br>Bigdata.com tool &nbsp;`get_market_tearsheet`

---

**[3]** Nasdaq 100 Flashes Warning Signs After 5-Day Slide: Is The Tech Rally Running Out Of Steam?<br>Benzinga &nbsp;|&nbsp; Jun 29, 2026<br>[https://www.benzinga.com/node/60149061?utm_campaign=partner_feed&utm_m](https://www.benzinga.com/node/60149061?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)

> *The 10 best-performing stocks in the Nasdaq 100 in 2026. This structural anxiety is compounded by immense sector concentration, with semiconductor giants like SanDisk Corp. (NASDAQ:SNDK), Micron Technology Inc. (NASDAQ:MU), and Intel Corp. (NASDAQ:INTC) completely dominating the index's top performe...*

> *The 10 best-performing stocks in the Nasdaq 100 in 2026. Read Also: After The June Swoon, Could July Deliver A Breakout Rally? Carson Group Strategist Ryan Detrick Says This Rare S&P 500 Signal Has Never Failed Unprecedented Volatility Disconnect While tech indexes slip, options markets are flashing...*

---

**[4]** Forex Bulletin - 07/28/2026<br>Gedik Investment Securities &nbsp;|&nbsp; Jul 28, 2026<br>[https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8c](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/0a60d6e3-b956-4fed-9b0a-9ae2a9ebe157)

> *Nasdaq Near Month Nasdaq futures declined as ongoing sharp sell-offs in semiconductor stocks weakened risk appetite in the technology sector. Losses in Nvidia, AMD, Micron, Sandisk, and SK Hynix stocks stemmed from concerns about the sustainability of artificial intelligence investments and potentia...*

---

...

**[6]** Forex Bulletin - 07/29/2026<br>Gedik Investment Securities &nbsp;|&nbsp; Jul 29, 2026<br>[https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8c](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/cc1da32e-fd38-4677-a80c-ab35b23f18e9)

> *Nasdaq Near Month Nasdaq futures are showing weakness as tensions in the Middle East are driving up oil prices, which in turn are putting pressure on inflation and interest rate expectations once again. While the Fed is expected to keep interest rates steady, the possibility of a rate hike remains, ...*

---

**[7]** Zacks Investment Ideas feature highlights: Nasdaq 100 Index ETF, Microsoft, Meta Platforms, Tesla and Alphabet<br>Nasdaq &nbsp;|&nbsp; Jul 29, 2026<br>[https://www.nasdaq.com/articles/zacks-investment-ideas-feature-highlig](https://www.nasdaq.com/articles/zacks-investment-ideas-feature-highlights-nasdaq-100-index-etf-microsoft-meta-platforms)

> *For Immediate Release Chicago, IL - July 29, 2026 - Today, Zacks Investment Ideas feature highlights Nasdaq 100 Index ETF QQQ, Microsoft MSFT, Meta Platforms META, Tesla TSLA and Alphabet GOOGL. Wall Street Rebounds Ahead of FOMC & Big Tech Earnings Tuesday, the wild summer volatility continued on W...*

---

**[8]** Forex Bulletin - 07/27/2026<br>Gedik Investment Securities &nbsp;|&nbsp; Jul 27, 2026<br>[https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8c](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/ef7bccaf-9116-428c-819d-eb50816d7c32)

> *Nasdaq Near Term Nasdaq futures started the week with an increase as attacks between the US and Iran ceased and oil prices retreated. For tech stocks, announcements regarding Microsoft, Meta, and Apple earnings, as well as artificial intelligence spending, will be decisive for the index's direction....*

---

**[9]** Forex Bulletin - 07.22.2026<br>Gedik Investment Securities &nbsp;|&nbsp; Jul 22, 2026<br>[https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8c](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/c899ba42-0437-4a7c-b22e-4338337882f5)

> *Nasdaq Near Term The earnings season in the US and the strong financial results of technology companies announced yesterday have supported the positive divergence of the Nasdaq index. The Nasdaq index closed with a gain of over 1% yesterday evening. Today, the financial results of Alphabet and Tesla...*

---

**[10]** Forex Bulletin - July 23, 2026<br>Gedik Investment Securities &nbsp;|&nbsp; Jul 23, 2026<br>[https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8c](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/375ee662-ccff-4931-9063-c013e695aeaf)

> *Nasdaq Near Term Earnings season in the US and reported profits can cause stock-sector specific divergences. On Tuesday, the Nasdaq index, which recorded an increase due to strong earnings, closed yesterday with a slight loss. Today, Intel (INTC) will announce its financial results. Intraday technic...*

---

### Verify: no blank references

Every entry must resolve to either a document headline or a tool title.

In [55]:
blank = [
    c
    for c in numbered_citations
    if not (c.get("headline") or c.get("title") or c.get("tool_name"))
]
tool_level = [c for c in numbered_citations if c.get("type") == "TOOL"]

print(f"total references:  {len(numbered_citations)}")
print(f"  documents:       {len(numbered_citations) - len(tool_level)}")
print(f"  tool-level:      {len(tool_level)}")
print(f"  blank ('N/A'):   {len(blank)}")
assert not blank, f"unresolved references: {blank}"
print("\nOK - every reference resolved.")

total references:  10
  documents:       9
  tool-level:      1
  blank ('N/A'):   0

OK - every reference resolved.


### Markdown export with a Sources section

`get_markdown_with_citations()` returns a self-contained document: the annotated answer plus a deduplicated Sources list, formatted as `Source name - MMM DD, YYYY` and linked to the canonical URL.

In [56]:
markdown_report = result.get_markdown_with_citations()

with open("output/spx_report.md", "w") as fh:
    fh.write(markdown_report)

print(markdown_report[markdown_report.index("## Sources") :])

## Sources

1. [Benzinga - Jul 28, 2026](https://www.benzinga.com/node/60736942?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack) - Nasdaq 100 Enters Correction, SanDisk Sinks Over 50% From Peak: Stock Market Today
2. Market Tearsheet - Bigdata.com `get_market_tearsheet`
3. [Benzinga - Jun 29, 2026](https://www.benzinga.com/node/60149061?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack) - Nasdaq 100 Flashes Warning Signs After 5-Day Slide: Is The Tech Rally Running Out Of Steam?
4. [Gedik Investment Securities - Jul 28, 2026](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/0a60d6e3-b956-4fed-9b0a-9ae2a9ebe157) - Forex Bulletin - 07/28/2026
5. [Gedik Investment Securities - Jul 24, 2026](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/607d8f3d-89ce-42f3-a419-ed88cace75c0) - Forex Bulletin - July 24, 2026
6. [Gedik Investment Securities - Jul 29, 2026](https://research.bluematrix.com/docs/pdf/90be6b15

---
## 5. Charts

With `chart_generation=True` the agent may run Python over Bigdata's structured data and return a `CHART` event carrying a [Vega-Lite](https://vega.github.io/vega-lite/) spec, plus `start`/`end` offsets pointing at the answer span the chart illustrates.

JupyterLab renders Vega-Lite natively from a MIME bundle, so no extra plotting dependency is needed. The bundle below is published under both the v5 and v6 media types, because front-ends only render a media type they have a registered renderer for and v6 support is still rolling out. Any other Vega-Lite renderer works equally well on `chart.vega_lite_spec`.

In [57]:
from research_client import Chart


def render_chart(chart: Chart) -> None:
    """Display a CHART event's Vega-Lite spec, and save the spec alongside it."""
    display(Markdown(f"**{chart.title}** - {chart.caption or chart.chart_type}"))
    display(
        {
            "application/vnd.vegalite.v6+json": chart.vega_lite_spec,
            "application/vnd.vegalite.v5+json": chart.vega_lite_spec,
            "text/plain": f"<Vega-Lite {chart.chart_type} chart: {chart.title}>",
        },
        raw=True,
    )

    path = f"output/{chart.chart_id}.vl.json"
    with open(path, "w") as fh:
        json.dump(chart.vega_lite_spec, fh, indent=2)
    print(f"spec saved to {path}")


if result.charts:
    for chart in result.charts:
        print(f"{chart.chart_id}: {chart.data_points} points, answer span {chart.start}-{chart.end}")
        render_chart(chart)
else:
    print("No charts in this response.")
    print("The agent draws one only when it judges a chart adds to the answer;")
    print("ask for it explicitly to force the code execution path (see below).")

No charts in this response.
The agent draws one only when it judges a chart adds to the answer;
ask for it explicitly to force the code execution path (see below).


### Force a chart

Asking for a chart directly makes the agent run code and emit a `CHART` event.

In [46]:
chart_result = client.research(
    message="Chart the S&P 500 (SPX) index level over the last 30 days, compare it with Apple, Google, and Microsoft.",
    research_effort="lite",
    chart_generation=True,
    on_event=make_progress_printer(),
)

print(f"\ncharts returned: {len(chart_result.charts)}")
for chart in chart_result.charts:
    render_chart(chart)

  action: search_companies
  action: search
  action: search_companies
  action: search_companies
  action: python_code_execution
  action: python_code_execution
  action: python_code_execution
  chart: Performance Comparison (Last 30 Days) (line)

charts returned: 1


**Performance Comparison (Last 30 Days)** - Performance Comparison (Last 30 Days)

<Vega-Lite line chart: Performance Comparison (Last 30 Days)>

spec saved to output/chart_67a954508cb7.vl.json


---
## 6. Other output formats

| Method | Returns |
|--------|---------|
| `get_answer()` | Plain answer text |
| `get_answer_with_citations()` | Answer with `[1]`, `[2, 3]` markers |
| `get_markdown_with_citations()` | Answer plus a Sources section |
| `get_numbered_citations()` | Citations numbered to match the inline markers |
| `get_citations()` / `get_citations_json()` | Every document returned by search |
| `to_dict()` / `to_json()` | Full result |
| `to_dict_with_inline_citations()` / `to_json_with_inline_citations()` | Full result, answer annotated |

Pass `include_tool_citations=False` to any of the citation methods for a document-only reference list.

In [58]:
# Plain answer, no markers
print(result.get_answer()[:400], "...\n")

# Document citations only, in Bigdata.com citation format
citations = result.get_citations()
print(f"{len(citations)} documents collected from search\n")
print(json.dumps(citations[0], indent=2)[:900], "...")

### Executive Summary
The Nasdaq 100 is currently experiencing a period of significant volatility and has officially slipped into correction territory, having fallen more than 10% from its record high set in early June 2026 . While the broader market has shown resilience, the Nasdaq 100 faces headwinds from a sharp rotation out of the semiconductor and AI-hardware sectors, which had previously bee ...

19 documents collected from search

{
  "id": "CA2C793CA066603E7E1E80165BE19EC8",
  "headline": "Nasdaq 100 Enters Correction, SanDisk Sinks Over 50% From Peak: Stock Market Today",
  "timestamp": "2026-07-28T17:21:06",
  "source": {
    "id": "5A5702",
    "name": "Benzinga",
    "rank": "RANK_1"
  },
  "url": "https://www.benzinga.com/node/60736942?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack",
  "source_type": "BIGDATA",
  "chunks": [
    {
      "cnum": 1,
      "text": "The Nasdaq 100 officially slipped into correction territory on Tuesday, falling more than 10% fr

---
## 7. Save results

In [59]:
outputs = {
    "output/citations.json": result.get_citations_json(),
    "output/research_result.json": result.to_json(),
    "output/result_with_inline_citations.json": result.to_json_with_inline_citations(),
}

for path, content in outputs.items():
    with open(path, "w") as fh:
        fh.write(content)
    print(f"saved {path} ({len(content):,} bytes)")

saved output/citations.json (37,953 bytes)
saved output/research_result.json (42,615 bytes)
saved output/result_with_inline_citations.json (19,376 bytes)


---
## 8. Follow-up questions

`follow_up()` reuses the previous result's `chat_id`, so the agent keeps the earlier turns in context. This requires `persistence_mode="enabled"` on the client; with the API default of `"disabled"` no history is saved and the follow-up starts cold.

In [60]:
follow_up_result = client.follow_up(
    "Which sectors are driving that performance?",
    previous_result=result,
    research_effort="lite",
)

print(f"same conversation: {follow_up_result.chat_id == result.chat_id}\n")
display(Markdown(follow_up_result.get_answer_with_citations()))

same conversation: True



The recent performance of the Nasdaq 100—specifically its slide into correction territory—has been driven by a sharp divergence between the cooling semiconductor/AI-hardware sector and the broader mega-cap technology complex.

### 1. The Primary Drag: Semiconductors and Memory
The semiconductor and memory sectors are the leading drivers of the index's current downward momentum. After serving as the engines of growth earlier in the year, this sector has undergone a significant "valuation reset" due to several factors:
*   **Sector-Specific Selling:** The memory sector has been at the heart of July's rout, with significant pullbacks in companies like SanDisk Corp. (SNDK) and Micron Technology (MU) [1].
*   **Sentiment Shift:** Investors are increasingly concerned about the sustainability of AI-related capital expenditures and rising international competition, particularly from China, which has led to a rapid exit from these stocks [1, 2]. 
*   **Equipment Manufacturers:** Broader weakness has extended to semiconductor equipment manufacturers such as Applied Materials and Lam Research, which have contributed to the index’s recent decline [3].

### 2. The Fulcrum of Volatility: Mega-Cap Tech ("Magnificent Seven")
While the semiconductor sector is driving the current selling pressure, the "Magnificent Seven" (mega-cap tech) stocks remain the dominant force dictating the index's overall sentiment and volatility. 
*   **Concentration Risk:** Because the Nasdaq 100 is highly concentrated in these names, their performance acts as the primary fulcrum for the index [3]. 
*   **Earnings Sensitivity:** Currently, investors are rotating their focus toward upcoming earnings reports from giants like Microsoft, Meta, Alphabet, and Apple to determine whether the broader tech rally can find a floor or if valuation concerns will trigger further profit-taking [1, 2, 4]. 

In essence, while the semiconductor sector is currently the "detractor" driving the correction, mega-cap tech remains the "driver" of the index's overall direction and market sentiment.

In [63]:
numbered_citations = follow_up_result.get_numbered_citations()

display(Markdown(f"---\n## References ({len(numbered_citations)})\n"))
print_citation(numbered_citations)

---
## References (4)


**[1]** Nasdaq 100 Enters Correction, SanDisk Sinks Over 50% From Peak: Stock Market Today<br>Benzinga &nbsp;|&nbsp; Jul 28, 2026<br>[https://www.benzinga.com/node/60736942?utm_campaign=partner_feed&utm_m](https://www.benzinga.com/node/60736942?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)

> *The Nasdaq 100 officially slipped into correction territory on Tuesday, falling more than 10% from its early June record high as the AI-driven sell-off deepened on mounting concerns over rising competition from China. The memory sector has been at the heart of July's rout. SanDisk Corp. (NASDAQ:SNDK...*

---

**[2]** Forex Bulletin - 07/28/2026<br>Gedik Investment Securities &nbsp;|&nbsp; Jul 28, 2026<br>[https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8c](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/0a60d6e3-b956-4fed-9b0a-9ae2a9ebe157)

> *Nasdaq Near Month Nasdaq futures declined as ongoing sharp sell-offs in semiconductor stocks weakened risk appetite in the technology sector. Losses in Nvidia, AMD, Micron, Sandisk, and SK Hynix stocks stemmed from concerns about the sustainability of artificial intelligence investments and potentia...*

---

**[3]** Nasdaq 100 Flashes Warning Signs After 5-Day Slide: Is The Tech Rally Running Out Of Steam?<br>Benzinga &nbsp;|&nbsp; Jun 29, 2026<br>[https://www.benzinga.com/node/60149061?utm_campaign=partner_feed&utm_m](https://www.benzinga.com/node/60149061?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)

> *The 10 best-performing stocks in the Nasdaq 100 in 2026. This structural anxiety is compounded by immense sector concentration, with semiconductor giants like SanDisk Corp. (NASDAQ:SNDK), Micron Technology Inc. (NASDAQ:MU), and Intel Corp. (NASDAQ:INTC) completely dominating the index's top performe...*

---

**[4]** Forex Bulletin - July 23, 2026<br>Gedik Investment Securities &nbsp;|&nbsp; Jul 23, 2026<br>[https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8c](https://research.bluematrix.com/docs/pdf/90be6b15-c069-45dc-b7d1-ddf8ce63aad6/375ee662-ccff-4931-9063-c013e695aeaf)

> *Nasdaq Near Term Earnings season in the US and reported profits can cause stock-sector specific divergences. On Tuesday, the Nasdaq index, which recorded an increase due to strong earnings, closed yesterday with a slight loss. Today, Intel (INTC) will announce its financial results. Intraday technic...*

---

---
## 9. Error handling

Failures arrive in two layers, and they need different treatment.

**HTTP level** - raised before the stream starts, mapped to typed exceptions:

| Status | Exception | Retryable |
|-------:|-----------|-----------|
| `400` / `422` | `InvalidRequestError` | No |
| `401` | `AuthenticationError` | No |
| `403` | `EntitlementError` | No |
| `404` | `ResourceNotFoundError` | No |
| `429` | `RateLimitError` | Yes, with backoff |
| `5xx` | `ServerError` | Yes, with backoff |

**Stream level** - typed messages inside a `200` response:

| Message | Handling |
|---------|----------|
| `LLM_RETRY` | Logged. The agent recovers on its own |
| `TOOL_ERROR` | Counted in `result.tool_errors`, never raised. Check it to detect a degraded answer |
| `ERROR` | Raised as `StreamError`. The stream is over |
| No `COMPLETE` | Raised as `TruncatedStreamError`, so a truncated stream never looks like an empty answer |

All of these derive from `ResearchAgentError`.

In [50]:
from research_client import ResearchAgentError

# A rejected credential surfaces as a typed exception before the stream starts,
# rather than as a silently empty answer. The API answers 401 or 403 depending
# on whether the key is malformed or simply unknown.
try:
    ResearchClient(api_key="not-a-real-key").research("test", research_effort="lite")
except ResearchAgentError as exc:
    print(f"{type(exc).__name__}: {exc}")

EntitlementError: Not entitled to this resource (403): {"status_code": 403, "message": "Authorization failed for this resource.", "request_id": "019faec9-b389-74e4-96bf-63ceab380062"}


In [51]:
# A degraded answer is visible without inspecting the stream.
if result.tool_errors:
    print("Some sources could not be retrieved; the answer may be incomplete.")
    for tool, count in result.tool_errors.items():
        print(f"  {tool}: {count} failure(s)")
else:
    print("No tool errors - every source was retrieved.")

No tool errors - every source was retrieved.


---
## Citation format reference

Document citations follow the standard Bigdata.com format. Only non-null fields are included.

```json
{
  "number": 2,
  "id": "80CBFA19A18071707AADAC6765E1F9F0",
  "headline": "SC SECTOR WATCH",
  "timestamp": "2026-07-27T10:32:26",
  "source": {"id": "7ED2AF", "name": "S&P Capital IQ NetAdvantage", "rank": "RANK_2"},
  "url": "https://...",
  "source_type": "BIGDATA",
  "chunks": [
    {"cnum": 5, "text": "Relevant excerpt...", "relevance": 0.94, "sentiment": 0.82}
  ]
}
```

Whole-tool attributions use a distinct shape, flagged by `"type": "TOOL"`:

```json
{
  "number": 1,
  "type": "TOOL",
  "tool_name": "get_market_tearsheet",
  "audit_id": "action_70282a87",
  "title": "Market Tearsheet"
}
```

### Source shapes

`source` on a grounding reference is one of two shapes, discriminated on `type`:

| | `BIGDATA` | `EXTERNAL` |
|---|---|---|
| Publisher name | `src_name` | `action.name` |
| URL | `url` | `action.url` |
| Headline | `hd` | `hd` |
| Date | `ts` | `ts` |

Reading only the `BIGDATA` field names is why external web results used to appear without a publisher or link. The client normalises both shapes onto the same `Citation` fields.

### Deduplication

Sources are deduplicated by `id`, falling back to `url`, then `hd`. Headlines are the last resort because unrelated documents can share a title - deduplicating on them merges distinct articles into one reference.